In [1]:
from ultralytics import YOLO
import cv2

In [2]:
model = YOLO("16_09_25_SENd_v2.5_10s_960.pt")

WARNING 16_09_25_SENd_v2.5_10s_960.pt appears to require 'dill', which is not in Ultralytics requirements.
AutoInstall will run now for 'dill' but this feature will be removed in the future.
Recommend fixes are to train a new model using the latest 'ultralytics' package or to run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo11n.pt'
requirements: Ultralytics requirement ['dill'] not found, attempting AutoUpdate...

requirements: AutoUpdate success  6.1s
WARNING requirements: Restart runtime or rerun command for updates to take effect



In [ ]:
cap = cv2.VideoCapture("recordings/camera_4.mp4")
frame_id = 0
while True:
    ret,frame = cap.read()
    if not ret:
        print("no rect")
        break

    results = model(frame, verbose = False)
    for result in results:
        for box in result.boxes:
            cls = int(box.cls[0].cpu().numpy())

            if cls not in [0,1,2]:
                print("no cls")
                continue
            label = model.names[cls]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cx = int((x1 + x2) / 2)
            cy = int(y2)
            print(f"Frame {frame_id} | Class: {label} ({cls}) | Bottom-center → ({cx}, {cy})")

    frame_id += 1

cap.release()


hi
Frame 0 | Class: forklift (1) | Bottom-center → (945, 279)
hi
Frame 1 | Class: forklift (1) | Bottom-center → (945, 280)
hi
Frame 2 | Class: forklift (1) | Bottom-center → (945, 280)
hi
Frame 3 | Class: forklift (1) | Bottom-center → (945, 280)
hi
Frame 4 | Class: forklift (1) | Bottom-center → (945, 280)


KeyboardInterrupt: 

In [ ]:

video_paths = [
    "recordings/camera_1.mp4",
    "recordings/camera_2.mp4",
    "recordings/camera_3.mp4",
    "recordings/camera_4.mp4"
]

caps = [cv2.VideoCapture(p) for p in video_paths]
frame_id = 0

while True:
    frames = []
    valid_read = True

    for i, cap in enumerate(caps):
        ret, frame = cap.read()
        if not ret:
            print(f"Video {i+1} ended.")
            valid_read = False
            break
        frames.append(frame)

    if not valid_read:
        print("All videos processed.")
        break

    print(f"\n Processing Frame {frame_id} from all videos")
    for i, frame in enumerate(frames):
        results = model(frame, verbose=False)
        print(f"Video {i+1}")

        for result in results:
            for box in result.boxes:
                cls = int(box.cls[0].cpu().numpy())
                if cls not in [0, 1, 2]:
                    continue

                label = model.names[cls]
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                cx = int((x1 + x2) / 2)
                cy = int(y2)

                print(f"Video {i+1} | Frame {frame_id} | Class: {label} ({cls}) | Bottom-center → ({cx}, {cy})")

    frame_id += 1

for cap in caps:
    cap.release()

print("Done with all videos.")



 Processing Frame 0 from all videos
Video 1
Video 2
Video 3
Video 4
Video 4 | Frame 0 | Class: forklift (1) | Bottom-center → (945, 279)

 Processing Frame 1 from all videos
Video 1
Video 2
Video 3
Video 4
Video 4 | Frame 1 | Class: forklift (1) | Bottom-center → (945, 280)

 Processing Frame 2 from all videos
Video 1
Video 2
Video 3
Video 4
Video 4 | Frame 2 | Class: forklift (1) | Bottom-center → (945, 280)

 Processing Frame 3 from all videos
Video 1
Video 2
Video 3


KeyboardInterrupt: 

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import open3d as o3d
import threading
from queue import Queue
import time

fisheye_params = {
    "width": 2592,
    "height": 1944,
    "cx": 1296.0,
    "cy": 972.0,
    "radius": 952.56,
    "theta_max_rad": 1.5708
}

cam_settings = [
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": -18.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 54.0, "yaw": 90.0-42.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 28.0, "yaw": 180.0-30.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": 270.0 + 24.0},
]

video_paths = [
    "recordings/camera_1.mp4",
    "recordings/camera_2.mp4",
    "recordings/camera_3.mp4",
    "recordings/camera_4.mp4"
]
fisheye_path = "recordings/fisheye.mp4"

#undistort to BEV
BEV = cv2.imread("ohlf.png")   
homography_BEV = np.load("Homography for 229_camera_.npy")
dis_rect_w, dis_rect_h = 2592,2592
fov = 160.0
fov_RAD = np.deg2rad(fov)
focal = (dis_rect_w / 2) / np.tan(fov_RAD / 2)

dis_K_rect = np.array([[focal, 0.0, dis_rect_w / 2],
                   [0.0, focal, dis_rect_h / 2],
                   [0.0, 0.0, 1.0]])

f_fish = fisheye_params["radius"] / fisheye_params["theta_max_rad"]
K_fish = np.array([[f_fish, 0.0, fisheye_params["cx"]],
                   [0.0, f_fish, fisheye_params["cy"]],
                   [0.0, 0.0, 1.0]])
D_fish = np.zeros((4, 1)) 

#3d rendering
added_spheres = []
point_queue = Queue()
homography_BEV_to_3D = np.load("Homography for bev to 3d.npy")
mesh = o3d.io.read_triangle_mesh("OHLF_obj/OHLF_v2.8.3 (1).obj", enable_post_processing=True)
mesh.compute_vertex_normals()
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="3D Model", width=1280, height=720)
vis.add_geometry(mesh)
opt = vis.get_render_option()


def load_images(views, fisheye_path=None):
    image_bgr = []
    for p in views:
        b = cv2.imread(p)
        if b is None:
            raise FileNotFoundError(f"Missing {p}")
        image_bgr.append(b)
    fisheye_img = None
    if fisheye_path:
        fisheye_img = cv2.imread(fisheye_path)
        
    return image_bgr, fisheye_img
views, fisheye_img = load_images(views_path, fisheye_path)
rect_h, rect_w = views[0].shape[:2]

def compute_basis(yaw_deg, pitch_deg):
    yaw_deg_corrected = yaw_deg - 90.0
    yaw, pitch = np.deg2rad([yaw_deg_corrected, pitch_deg])

    Rz = np.array([
        [ np.cos(-yaw), -np.sin(-yaw), 0],
        [ np.sin(-yaw),  np.cos(-yaw), 0],
        [ 0,             0,            1]
    ])
    Rx = np.array([
        [ 1, 0,           0          ],
        [ 0, np.cos(pitch), -np.sin(pitch)],
        [ 0, np.sin(pitch),  np.cos(pitch)]
    ])
    R_ = Rx @ Rz
    R = np.diag([1, -1, 1]) @ R_
    return R

def rect_to_fisheye_point(u, v, view_params, fisheye_params):
    Hr, Wr = rect_h, rect_w
    cx_r, cy_r = Wr / 2, Hr / 2
    fov = view_params["fov"]
    f = (Wr / 2) / np.tan(np.deg2rad(fov / 2))
    x = (u - cx_r) / f
    y = (v - cy_r) / f
    z = 1
    ray_rect = np.array([x, y, z])
    ray_rect /= np.linalg.norm(ray_rect)
    Rot = compute_basis(view_params["yaw"], view_params["pitch"])
    ray_fish = Rot.T @ ray_rect
    Xf, Yf, Zf = ray_fish
    theta = np.arccos(np.clip(Zf, -1, 1))
    phi = np.arctan2(Yf, Xf)
    radius, theta_max = fisheye_params["radius"], fisheye_params["theta_max_rad"]
    cx_f, cy_f = fisheye_params["cx"], fisheye_params["cy"]
    r = (theta / theta_max) * radius
    u_f = cx_f + r * np.cos(phi)
    v_f = cy_f - r * np.sin(phi)
    return (int(u_f), int(v_f))

def fisheye_to_rect_point(u_f, v_f, view_params, fisheye_params):
    cx_f, cy_f = fisheye_params["cx"], fisheye_params["cy"]
    radius, theta_max = fisheye_params["radius"], fisheye_params["theta_max_rad"]

    dx, dy = u_f - cx_f, cy_f - v_f
    r = np.sqrt(dx ** 2 + dy ** 2)
    theta = (r / radius) * theta_max
    phi = np.arctan2(dy, dx)
    sin_t = np.sin(theta)
    ray_fish = np.array([sin_t * np.cos(phi), 
                         sin_t * np.sin(phi), 
                         np.cos(theta)])
    Rot = compute_basis(view_params["yaw"], view_params["pitch"])
    ray_rect = Rot @ ray_fish

    Xc, Yc, Zc = ray_rect
    if Zc <= 0:
        return None

    Hr, Wr = rect_h, rect_w
    fov = view_params["fov"]
    f = (Wr / 2) / np.tan(np.deg2rad(fov / 2))
    cx_r, cy_r = Wr / 2, Hr / 2
    u = f * (Xc / Zc) + cx_r
    v = f * (Yc / Zc) + cy_r
    if 0 <= u < Wr and 0 <= v < Hr:
        return (int(u), int(v))
    return None

def make_rect_grid_disp():
    grid = np.ones((RECT_TILE_H_DISP * 2, RECT_TILE_W_DISP * 2, 3), dtype=np.uint8)
    grid[0:RECT_TILE_H_DISP, 0:RECT_TILE_W_DISP] = views_disp[0]
    grid[0:RECT_TILE_H_DISP, RECT_TILE_W_DISP:] = views_disp[1]
    grid[RECT_TILE_H_DISP:, 0:RECT_TILE_W_DISP] = views_disp[2]
    grid[RECT_TILE_H_DISP:, RECT_TILE_W_DISP:] = views_disp[3]
    return grid

def map_2d_to_3d(u, v, homography_bev_to_3d, y_value=0.0):
    src = np.array([u, v, 1.0], dtype=np.float64)
    dst = homography_bev_to_3d @ src
    dst /= dst[2]
    return np.array([dst[0], y_value, dst[1]])

def add_point_realtime(point3d):
    if added_spheres:
        vis.remove_geometry(added_spheres[-1], reset_bounding_box=False)
        added_spheres.clear()

    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.25)
    sphere.paint_uniform_color([1, 0, 0])
    sphere.translate(point3d)
    vis.add_geometry(sphere, reset_bounding_box=False)
    vis.update_geometry(sphere)
    vis.poll_events()
    vis.update_renderer()
    added_spheres.append(sphere)



#Display
scale_BEV=0.5
scale_rect = 0.5
FISH_W_DISP, FISH_H_DISP = 1280, 720
RECT_TILE_W_DISP, RECT_TILE_H_DISP = int(rect_w * scale_rect), int(rect_h * scale_rect)
bev_display = cv2.resize(BEV, (0,0), fx=scale_BEV, fy=scale_BEV)
views_disp = [cv2.resize(v, (0,0), fx=scale_rect, fy=scale_rect) for v in views]
fisheye_display = cv2.resize(fisheye_img, (FISH_W_DISP, FISH_H_DISP))

scale_fx = fisheye_params["width"] / FISH_W_DISP
scale_fy = fisheye_params["height"] / FISH_H_DISP
rect_grid_disp = make_rect_grid_disp()

def draw_text(img, text, pos, color=(255, 255, 255)):
    font = cv2.FONT_HERSHEY_COMPLEX  
    scale = 0.8                 
    thickness = 2         
    x, y = int(pos[0]), int(pos[1])
    cv2.putText(img, text, (x + 1, y + 1), font, scale, (0, 0, 0), thickness , cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

def run():
    def on_mouse(event, x, y, flags, param):
        global canvas
        if event != cv2.EVENT_LBUTTONDOWN:
            return

        fisheye_disp = fisheye_display.copy()
        rect_disp = rect_grid_disp.copy()
        bev_disp = bev_display.copy()

        if x < FISH_W_DISP:  
            u_full, v_full = x * scale_fx, y * scale_fy
            cv2.circle(fisheye_disp, (x, y), 3, (0, 255, 0), -1)
            draw_text(fisheye_disp, f"({u_full:.1f},{v_full:.1f})", (x, y), (0, 255, 0))
            print(f"Click Fisheye Point = ({u_full:.1f},{v_full:.1f})")

            for i, cam in enumerate(cam_settings):
                pt = fisheye_to_rect_point(u_full, v_full, cam, fisheye_params)
                if pt is not None:
                    u_rect, v_rect = pt
                    ox = (i % 2) * RECT_TILE_W_DISP
                    oy = (i // 2) * RECT_TILE_H_DISP
                    u_disp, v_disp = int(u_rect * scale_rect), int(v_rect * scale_rect)
                    pos = (ox + u_disp, oy + v_disp)
                    cv2.circle(rect_disp, pos, 3, (0, 0, 255), -1)
                    draw_text(rect_disp, f"({u_rect},{v_rect})", pos, (0, 0, 255))
                    print(f" Rect view {i+1}: Rect Point = ({u_rect:.1f},{v_rect:.1f}) ")

        else: 
            xr = x - FISH_W_DISP
            xi = xr % RECT_TILE_W_DISP
            yi = y % RECT_TILE_H_DISP
            view_idx = (y >= RECT_TILE_H_DISP) * 2 + (xr >= RECT_TILE_W_DISP)
            u_full, v_full = xi / scale_rect, yi / scale_rect
            print(f"Click Rect_point = ({u_full},{v_full}) ")
            draw_text(rect_disp, f"({u_full},{v_full})", (x - FISH_W_DISP, y), (0, 255, 0))
            cv2.circle(rect_disp, (x - FISH_W_DISP, y), 3, (0, 255, 0), -1)
            pt_f = rect_to_fisheye_point(u_full, v_full, cam_settings[view_idx], fisheye_params)
            if pt_f is not None:
                u_f, v_f = pt_f
                u_fd, v_fd = int(u_f / scale_fx), int(v_f / scale_fy)
                cv2.circle(fisheye_disp, (u_fd, v_fd), 3, (0, 0,255), -1)
                draw_text(fisheye_disp, f"({int(u_f)},{int(v_f)})", (u_fd, v_fd), (0, 0, 255))
                pt_f = np.array(pt_f)
                pt_f = pt_f.astype(np.float32).reshape(-1,1,2)
                rect_point = cv2.fisheye.undistortPoints(pt_f, K_fish, D_fish, P=dis_K_rect)
                mapped_pt = cv2.perspectiveTransform(rect_point, homography_BEV)
                mx, my = int(mapped_pt[0][0][0]), int(mapped_pt[0][0][1])
                x_disp, y_disp = int(mx * scale_BEV), int(my * scale_BEV)
                cv2.circle(bev_disp, (x_disp, y_disp), 3, (0, 0, 255), -1)
                cv2.putText(bev_disp, f"({mx}, {my})", (x_disp + 10, y_disp - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
                p3d = map_2d_to_3d(mx,my,homography_BEV_to_3D)
                p3d = [abs(v) for v in p3d]
                point_queue.put(p3d)
                print(f" Fisheye Point = ({int(u_f)},{int(v_f)}) | BEV Point = ({mx},{my}) | 3D Model = ({p3d[0]:.2f},{p3d[1]:.2f},{p3d[2]:.2f})")
                
        
        canvas = np.ones((max(FISH_H_DISP, rect_disp.shape[0]), FISH_W_DISP + rect_disp.shape[1], 3), dtype=np.uint8)
        canvas[0:FISH_H_DISP, 0:FISH_W_DISP] = fisheye_disp
        canvas[0:rect_disp.shape[0], FISH_W_DISP:] = rect_disp
        # cv2.imshow("BEV Image", bev_disp)
        cv2.imshow("Unified Mapping", canvas)
        

    canvas = np.ones((rect_h, 2560, 3), dtype=np.uint8)
    canvas[0:rect_h, 0:rect_w] = fisheye_display
    canvas[0:rect_h, rect_w:] = rect_grid_disp

    cv2.namedWindow("Unified Mapping", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Unified Mapping", on_mouse)
    cv2.imshow("Unified Mapping", canvas)
    # cv2.imshow("BEV Image", bev_display)

    while True:
        if cv2.waitKey(20) & 0xFF == 27:
            break

    cv2.destroyAllWindows()

cv_thread = threading.Thread(target=run, daemon=True)
cv_thread.start()
try:
    while cv_thread.is_alive():
        while not point_queue.empty():
            p = point_queue.get()
            add_point_realtime(p)
        vis.poll_events()
        vis.update_renderer()
        time.sleep(0.03)
finally:
    vis.destroy_window()
    cv2.destroyAllWindows()

In [ ]:
import numpy as np
import cv2
import open3d as o3d
import threading
from queue import Queue
import time

# --- your params (unchanged) ---
fisheye_params = {
    "width": 2592,
    "height": 1944,
    "cx": 1296.0,
    "cy": 972.0,
    "radius": 952.56,
    "theta_max_rad": 1.5708
}

cam_settings = [
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": -18.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 54.0, "yaw": 90.0-42.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 28.0, "yaw": 180.0-30.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": 270.0 + 24.0},
]

video_paths = [
    "recordings/camera_1.mp4",
    "recordings/camera_2.mp4",
    "recordings/camera_3.mp4",
    "recordings/camera_4.mp4"
]
fisheye_path = "recordings/fisheye.mp4"

# BEV / homography / camera intrinsics
BEV = cv2.imread("ohlf.png")
homography_BEV = np.load("Homography for 229_camera_.npy")
dis_rect_w, dis_rect_h = 2592, 2592
fov = 160.0
fov_RAD = np.deg2rad(fov)
focal = (dis_rect_w / 2) / np.tan(fov_RAD / 2)

dis_K_rect = np.array([[focal, 0.0, dis_rect_w / 2],
                       [0.0, focal, dis_rect_h / 2],
                       [0.0, 0.0, 1.0]])

f_fish = fisheye_params["radius"] / fisheye_params["theta_max_rad"]
K_fish = np.array([[f_fish, 0.0, fisheye_params["cx"]],
                   [0.0, f_fish, fisheye_params["cy"]],
                   [0.0, 0.0, 1.0]])
D_fish = np.zeros((4, 1))

# 3D rendering
added_spheres = []
point_queue = Queue()
homography_BEV_to_3D = np.load("Homography for bev to 3d.npy")
mesh = o3d.io.read_triangle_mesh("OHLF_obj/OHLF_v2.8.3 (1).obj", enable_post_processing=True)
mesh.compute_vertex_normals()
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="3D Model", width=1280, height=720)
vis.add_geometry(mesh)

# --- load images (fix: use video_paths) ---
def load_videos(video_paths, fisheye_path=None, frame_index=0):

    views = []
    for path in video_paths:
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open {path}")
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # Clamp frame index to available range
        idx = min(frame_index, total_frames - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        cap.release()
        if not ret or frame is None:
            raise RuntimeError(f"Failed to read frame {idx} from {path}")
        views.append(frame)

    fisheye_frame = None
    if fisheye_path:
        cap_f = cv2.VideoCapture(fisheye_path)
        if not cap_f.isOpened():
            raise FileNotFoundError(f"Cannot open {fisheye_path}")
        total_frames_f = int(cap_f.get(cv2.CAP_PROP_FRAME_COUNT))
        idx_f = min(frame_index, total_frames_f - 1)
        cap_f.set(cv2.CAP_PROP_POS_FRAMES, idx_f)
        ret, frame_f = cap_f.read()
        cap_f.release()
        if ret and frame_f is not None:
            fisheye_frame = frame_f
        else:
            raise RuntimeError(f"Failed to read frame {idx_f} from fisheye video {fisheye_path}")

    return views, fisheye_frame

views, fisheye_img = load_videos(video_paths, fisheye_path, frame_index=0)
rect_h, rect_w = views[0].shape[:2]

# Display sizing constants (define BEFORE functions that use them)
scale_BEV = 0.5
scale_rect = 0.5
FISH_W_DISP, FISH_H_DISP = 1280, 720
RECT_TILE_W_DISP = int(rect_w * scale_rect)
RECT_TILE_H_DISP = int(rect_h * scale_rect)

# prepare display images
bev_display = cv2.resize(BEV, (0, 0), fx=scale_BEV, fy=scale_BEV)
views_disp = [cv2.resize(v, (0, 0), fx=scale_rect, fy=scale_rect) for v in views]
fisheye_display = cv2.resize(fisheye_img, (FISH_W_DISP, FISH_H_DISP))

scale_fx = fisheye_params["width"] / FISH_W_DISP
scale_fy = fisheye_params["height"] / FISH_H_DISP

def make_rect_grid_disp():
    # build a 2x2 grid of the four views
    grid_h = RECT_TILE_H_DISP * 2
    grid_w = RECT_TILE_W_DISP * 2
    grid = np.ones((grid_h, grid_w, 3), dtype=np.uint8) * 0
    grid[0:RECT_TILE_H_DISP, 0:RECT_TILE_W_DISP] = views_disp[0]
    grid[0:RECT_TILE_H_DISP, RECT_TILE_W_DISP:] = views_disp[1]
    grid[RECT_TILE_H_DISP:, 0:RECT_TILE_W_DISP] = views_disp[2]
    grid[RECT_TILE_H_DISP:, RECT_TILE_W_DISP:] = views_disp[3]
    return grid

rect_grid_disp = make_rect_grid_disp()

# --- utility drawing function (unchanged) ---
def draw_text(img, text, pos, color=(255, 255, 255)):
    font = cv2.FONT_HERSHEY_COMPLEX
    scale = 0.8
    thickness = 2
    x, y = int(pos[0]), int(pos[1])
    cv2.putText(img, text, (x + 1, y + 1), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

# --- keep your geometry mapping helpers (map_2d_to_3d etc.) ---
def map_2d_to_3d(u, v, homography_bev_to_3d, y_value=0.0):
    src = np.array([u, v, 1.0], dtype=np.float64)
    dst = homography_bev_to_3d @ src
    if abs(dst[2]) < 1e-8:
        return np.array([0.0, y_value, 0.0])
    dst /= dst[2]
    return np.array([dst[0], y_value, dst[1]])

# ----------------
# corrected run() with consistent coordinate handling
# ----------------
def run():
    def on_mouse(event, mx, my, flags, param):
        # only left clicks
        if event != cv2.EVENT_LBUTTONDOWN:
            return

        fisheye_disp = fisheye_display.copy()
        rect_disp = rect_grid_disp.copy()
        bev_disp = bev_display.copy()

        # click on fisheye area (left part)
        if mx < FISH_W_DISP:
            # convert display coords -> full-res fisheye coords
            u_full = mx * scale_fx
            v_full = my * scale_fy
            cv2.circle(fisheye_disp, (mx, my), 4, (0, 255, 0), -1)
            draw_text(fisheye_disp, f"({u_full:.1f},{v_full:.1f})", (mx + 6, my - 6), (0, 255, 0))
            print(f"Click Fisheye Point = ({u_full:.1f},{v_full:.1f})")

            for i, cam in enumerate(cam_settings):
                pt = fisheye_to_rect_point(u_full, v_full, cam, fisheye_params)
                if pt is not None:
                    u_rect, v_rect = pt  # rect in rect image pixel coordinates
                    # rectangle tile origin in display coords
                    ox = (i % 2) * RECT_TILE_W_DISP
                    oy = (i // 2) * RECT_TILE_H_DISP
                    u_disp = int(u_rect * scale_rect)
                    v_disp = int(v_rect * scale_rect)
                    pos = (ox + u_disp, oy + v_disp)
                    cv2.circle(rect_disp, pos, 3, (0, 0, 255), -1)
                    draw_text(rect_disp, f"({u_rect:.0f},{v_rect:.0f})", (pos[0] + 6, pos[1] - 6), (0, 0, 255))
                    print(f" Rect view {i+1}: Rect Point = ({u_rect:.1f},{v_rect:.1f}) ")

        # click on rect-grid area (right part)
        else:
            xr = mx - FISH_W_DISP                   # display-space x inside rect_grid_disp
            yr = my                                 # display-space y inside rect_grid_disp
            # make sure click lies inside rect grid
            if 0 <= xr < rect_disp.shape[1] and 0 <= yr < rect_disp.shape[0]:
                # display -> rect image pixel coordinates (per tile)
                xi_disp = int(xr)
                yi_disp = int(yr)
                # which tile
                view_idx = (yi_disp >= RECT_TILE_H_DISP) * 2 + (xi_disp >= RECT_TILE_W_DISP)
                # tile-local display coords
                tile_x_disp = xi_disp % RECT_TILE_W_DISP
                tile_y_disp = yi_disp % RECT_TILE_H_DISP
                # convert display to original rect image pixels
                u_full = tile_x_disp / scale_rect
                v_full = tile_y_disp / scale_rect

                # annotate the rect_disp at the clicked display position
                cv2.circle(rect_disp, (xi_disp, yi_disp), 4, (0, 255, 0), -1)
                draw_text(rect_disp, f"({u_full:.1f},{v_full:.1f})", (xi_disp + 6, yi_disp - 6), (0, 255, 0))
                print(f"Click Rect_point (view {view_idx}) = ({u_full:.1f},{v_full:.1f}) ")

                # compute corresponding fisheye point
                pt_f = rect_to_fisheye_point(u_full, v_full, cam_settings[view_idx], fisheye_params)
                if pt_f is not None:
                    u_f, v_f = pt_f
                    u_fd = int(u_f / scale_fx)
                    v_fd = int(v_f / scale_fy)
                    # mark on fisheye display
                    if 0 <= u_fd < FISH_W_DISP and 0 <= v_fd < FISH_H_DISP:
                        cv2.circle(fisheye_disp, (u_fd, v_fd), 3, (0, 0, 255), -1)
                        draw_text(fisheye_disp, f"({int(u_f)},{int(v_f)})", (u_fd + 6, v_fd - 6), (0, 0, 255))

                    # create undistorted rect point, perspective transform to BEV
                    pt_f_arr = np.array([[ [u_f, v_f] ]], dtype=np.float32)  # shape (1,1,2)
                    rect_point = cv2.fisheye.undistortPoints(pt_f_arr, K_fish, D_fish, P=dis_K_rect)
                    mapped_pt = cv2.perspectiveTransform(rect_point, homography_BEV)
                    mx_bev, my_bev = int(mapped_pt[0][0][0]), int(mapped_pt[0][0][1])
                    x_disp_bev, y_disp_bev = int(mx_bev * scale_BEV), int(my_bev * scale_BEV)
                    cv2.circle(bev_disp, (x_disp_bev, y_disp_bev), 3, (0, 0, 255), -1)
                    cv2.putText(bev_disp, f"({mx_bev},{my_bev})", (x_disp_bev + 8, y_disp_bev - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

                    p3d = map_2d_to_3d(mx_bev, my_bev, homography_BEV_to_3D)
                    p3d = [abs(v) for v in p3d]
                    point_queue.put(p3d)
                    print(f" Fisheye Point = ({int(u_f)},{int(v_f)}) | BEV Point = ({mx_bev},{my_bev}) | 3D Model = ({p3d[0]:.2f},{p3d[1]:.2f},{p3d[2]:.2f})")

        # composite canvas: left fisheye, right rect_grid
        h_canvas = max(fisheye_disp.shape[0], rect_disp.shape[0], bev_disp.shape[0])
        w_canvas = FISH_W_DISP + rect_disp.shape[1]
        canvas = np.ones((h_canvas, w_canvas, 3), dtype=np.uint8) * 0
        canvas[0:fisheye_disp.shape[0], 0:FISH_W_DISP] = fisheye_disp
        canvas[0:rect_disp.shape[0], FISH_W_DISP:FISH_W_DISP + rect_disp.shape[1]] = rect_disp

        # show
        cv2.imshow("Unified Mapping", canvas)
        # optional: show BEV in separate window
        cv2.imshow("BEV Image", bev_disp)

    # create window and mouse callback
    cv2.namedWindow("Unified Mapping", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Unified Mapping", on_mouse)

    # initial canvas
    h_canvas = max(fisheye_display.shape[0], rect_grid_disp.shape[0], bev_display.shape[0])
    w_canvas = FISH_W_DISP + rect_grid_disp.shape[1]
    canvas_init = np.ones((h_canvas, w_canvas, 3), dtype=np.uint8) * 0
    canvas_init[0:fisheye_display.shape[0], 0:FISH_W_DISP] = fisheye_display
    canvas_init[0:rect_grid_disp.shape[0], FISH_W_DISP:FISH_W_DISP + rect_grid_disp.shape[1]] = rect_grid_disp
    cv2.imshow("Unified Mapping", canvas_init)
    cv2.imshow("BEV Image", bev_display)

    while True:
        if cv2.waitKey(20) & 0xFF == 27:  # ESC to exit
            break

    cv2.destroyAllWindows()

# start threads / main loop (almost same as yours)
cv_thread = threading.Thread(target=run, daemon=True)
cv_thread.start()

try:
    while cv_thread.is_alive():
        while not point_queue.empty():
            p = point_queue.get()
            add_point_realtime(p)
        vis.poll_events()
        vis.update_renderer()
        time.sleep(0.03)
finally:
    vis.destroy_window()
    cv2.destroyAllWindows()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


FileNotFoundError: Missing recordings/camera_1.mp4

: 

In [1]:
import cv2

# --- Load the video ---
video_path = "recordings/camera_1.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Error: Cannot open video file.")
else:
    # --- Get frame width and height ---
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"✅ Frame size: {width} x {height}")

    # --- Optional: get FPS ---
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"🎞 FPS: {fps}")

cap.release()


✅ Frame size: 1280 x 720
🎞 FPS: 25.0


In [2]:
import numpy as np
import cv2
import open3d as o3d
import threading
from queue import Queue
import time
import sys

# -----------------------------
# CONFIG - edit paths & params
# -----------------------------
video_paths = [
    "recordings/camera_1.mp4",
    "recordings/camera_2.mp4",
    "recordings/camera_3.mp4",
    "recordings/camera_4.mp4"
]
fisheye_path = "recordings/fisheye.mp4"
bev_image_path = "ohlf.png"
homography_bev_path = "Homography for 229_camera_.npy"
homography_bev_to_3d_path = "Homography for bev to 3d.npy"
mesh_path = "OHLF_obj/OHLF_v2.8.3 (1).obj"

# fisheye intrinsic model (from your parameters)
fisheye_params = {
    "width": 2592,
    "height": 1944,
    "cx": 1296.0,
    "cy": 972.0,
    "radius": 952.56,
    "theta_max_rad": 1.5708
}

# approximate view parameters (yaw/pitch used in projection functions)
cam_settings = [
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": -18.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 54.0, "yaw": 48.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 28.0, "yaw": 150.0},
    {"cx": 1296.0, "cy": 972.0, "fov": 90.0, "pitch": 58.0, "yaw": 294.0},
]

# display sizing
scale_BEV = 0.5
scale_rect = 0.5
FISH_W_DISP, FISH_H_DISP = 1280, 720
frame_delay_ms = 50  # ms between frames

# -----------------------------
# Load static assets & intrinsics
# -----------------------------
BEV = cv2.imread(bev_image_path)
if BEV is None:
    print(f"ERROR: BEV image not found at {bev_image_path}", file=sys.stderr)
    raise FileNotFoundError(bev_image_path)

homography_BEV = np.load(homography_bev_path)
homography_BEV_to_3D = np.load(homography_bev_to_3d_path)

# dis_rect intrinsics for converting undistortPoints -> rectified intrinsics
dis_rect_w, dis_rect_h = 2592, 2592
fov_for_rect = 160.0
fov_RAD = np.deg2rad(fov_for_rect)
focal = (dis_rect_w / 2) / np.tan(fov_RAD / 2)
dis_K_rect = np.array([[focal, 0.0, dis_rect_w / 2],
                       [0.0, focal, dis_rect_h / 2],
                       [0.0, 0.0, 1.0]])

# fisheye intrinsics
f_fish = fisheye_params["radius"] / fisheye_params["theta_max_rad"]
K_fish = np.array([[f_fish, 0.0, fisheye_params["cx"]],
                   [0.0, f_fish, fisheye_params["cy"]],
                   [0.0, 0.0, 1.0]])
D_fish = np.zeros((4, 1))

# -----------------------------
# Open3D setup (3D model)
# -----------------------------
mesh = o3d.io.read_triangle_mesh(mesh_path, enable_post_processing=True)
mesh.compute_vertex_normals()
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="3D Model", width=1280, height=720)
vis.add_geometry(mesh)
added_spheres = []
point_queue = Queue()

def add_point_realtime(point3d):
    """Add single sphere in Open3D visualizer. Removes previous sphere."""
    if added_spheres:
        try:
            vis.remove_geometry(added_spheres[-1], reset_bounding_box=False)
        except Exception:
            pass
        added_spheres.clear()
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.25)
    sphere.paint_uniform_color([1.0, 0.0, 0.0])
    sphere.translate(point3d)
    vis.add_geometry(sphere, reset_bounding_box=False)
    vis.update_geometry(sphere)
    vis.poll_events()
    vis.update_renderer()
    added_spheres.append(sphere)

# -----------------------------
# Math / projection helpers
# -----------------------------
def compute_basis(yaw_deg, pitch_deg):
    """Rotation (rect camera frame <-> world/fisheye) using yaw/pitch."""
    yaw_deg_corrected = yaw_deg - 90.0
    yaw, pitch = np.deg2rad([yaw_deg_corrected, pitch_deg])
    Rz = np.array([
        [ np.cos(-yaw), -np.sin(-yaw), 0],
        [ np.sin(-yaw),  np.cos(-yaw), 0],
        [ 0, 0, 1]
    ])
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(pitch), -np.sin(pitch)],
        [0, np.sin(pitch),  np.cos(pitch)]
    ])
    R_ = Rx @ Rz
    # flip Y axis to match image coordinate conventions
    R = np.diag([1, -1, 1]) @ R_
    return R

def rect_to_fisheye_point(u, v, view_params, fisheye_params_local, rect_h_local, rect_w_local):
    """Project pixel (u,v) in rectified view -> fisheye image (u_f, v_f)."""
    Hr, Wr = rect_h_local, rect_w_local
    cx_r, cy_r = Wr / 2.0, Hr / 2.0
    f = (Wr / 2) / np.tan(np.deg2rad(view_params["fov"] / 2.0))

    x = (u - cx_r) / f
    y = (v - cy_r) / f
    ray_rect = np.array([x, y, 1.0])
    ray_rect = ray_rect / np.linalg.norm(ray_rect)

    Rot = compute_basis(view_params["yaw"], view_params["pitch"])
    ray_fish = Rot.T @ ray_rect
    Xf, Yf, Zf = ray_fish
    theta = np.arccos(np.clip(Zf, -1.0, 1.0))
    phi = np.arctan2(Yf, Xf)
    radius, theta_max = fisheye_params_local["radius"], fisheye_params_local["theta_max_rad"]
    cx_f, cy_f = fisheye_params_local["cx"], fisheye_params_local["cy"]
    r = (theta / theta_max) * radius
    u_f = cx_f + r * np.cos(phi)
    v_f = cy_f - r * np.sin(phi)
    return (int(round(u_f)), int(round(v_f)))

def fisheye_to_rect_point(u_f, v_f, view_params, fisheye_params_local, rect_h_local, rect_w_local):
    """Project pixel (u_f,v_f) in fisheye -> rectified view pixel or None if behind camera."""
    cx_f, cy_f = fisheye_params_local["cx"], fisheye_params_local["cy"]
    radius, theta_max = fisheye_params_local["radius"], fisheye_params_local["theta_max_rad"]
    dx, dy = u_f - cx_f, cy_f - v_f  # note sign to match earlier derivation
    r = np.sqrt(dx**2 + dy**2)
    if radius == 0:
        return None
    theta = (r / radius) * theta_max
    phi = np.arctan2(dy, dx)
    sin_t = np.sin(theta)
    ray_fish = np.array([sin_t * np.cos(phi), sin_t * np.sin(phi), np.cos(theta)])
    Rot = compute_basis(view_params["yaw"], view_params["pitch"])
    ray_rect = Rot @ ray_fish
    Xc, Yc, Zc = ray_rect
    if Zc <= 0:
        return None
    f = (rect_w_local / 2.0) / np.tan(np.deg2rad(view_params["fov"] / 2.0))
    cx_r, cy_r = rect_w_local / 2.0, rect_h_local / 2.0
    u = f * (Xc / Zc) + cx_r
    v = f * (Yc / Zc) + cy_r
    if 0 <= u < rect_w_local and 0 <= v < rect_h_local:
        return (int(round(u)), int(round(v)))
    return None

def map_2d_to_3d(u, v, homography_bev_to_3d_local, y_value=0.0):
    src = np.array([u, v, 1.0], dtype=np.float64)
    dst = homography_bev_to_3d_local @ src
    if abs(dst[2]) < 1e-8:
        return np.array([0.0, y_value, 0.0])
    dst /= dst[2]
    return np.array([dst[0], y_value, dst[1]])

def draw_text(img, text, pos, color=(255,255,255)):
    x, y = int(pos[0]), int(pos[1])
    cv2.putText(img, text, (x+1, y+1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1, cv2.LINE_AA)

# -----------------------------
# Video streaming & UI state
# -----------------------------
def open_video_caps(video_paths_local, fisheye_path_local):
    caps_local = []
    for p in video_paths_local:
        cap = cv2.VideoCapture(p)
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open {p}")
        caps_local.append(cap)
    fisheye_cap_local = None
    if fisheye_path_local:
        fisheye_cap_local = cv2.VideoCapture(fisheye_path_local)
        if not fisheye_cap_local.isOpened():
            raise FileNotFoundError(f"Cannot open {fisheye_path_local}")
    return caps_local, fisheye_cap_local

def get_next_frames(caps_local, fisheye_cap_local):
    frames = []
    for cap in caps_local:
        ret, frame = cap.read()
        if not ret:
            # loop video
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if not ret:
                frames.append(None)
                continue
        frames.append(frame)
    fisheye_frame = None
    if fisheye_cap_local:
        ret, frame = fisheye_cap_local.read()
        if not ret:
            fisheye_cap_local.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = fisheye_cap_local.read()
            if not ret:
                fisheye_frame = None
            else:
                fisheye_frame = frame
        else:
            fisheye_frame = frame
    return frames, fisheye_frame

# keep the current frame displays in a shared mutable dict so the mouse callback sees latest frames
state = {
    "rect_grid_disp": None,
    "fisheye_disp": None,
    "bev_disp": None,
    "rect_h": None,
    "rect_w": None,
    "RECT_TILE_W_DISP": None,
    "RECT_TILE_H_DISP": None,
    "scale_fx": None,
    "scale_fy": None
}

# -----------------------------
# Mouse callback: maps clicks
# -----------------------------
def on_mouse(event, mx, my, flags, param):
    if event != cv2.EVENT_LBUTTONDOWN:
        return
    # copy current display frames to avoid race
    fisheye_disp = state.get("fisheye_disp")
    rect_disp = state.get("rect_grid_disp")
    bev_disp = state.get("bev_disp")
    if fisheye_disp is None or rect_disp is None:
        print("No frames available yet.")
        return

    rect_h_local = state["rect_h"]
    rect_w_local = state["rect_w"]
    RECT_TILE_W_DISP = state["RECT_TILE_W_DISP"]
    RECT_TILE_H_DISP = state["RECT_TILE_H_DISP"]
    scale_fx = state["scale_fx"]
    scale_fy = state["scale_fy"]

    # make writable copies for drawing
    fisheye_copy = fisheye_disp.copy()
    rect_copy = rect_disp.copy()
    bev_copy = bev_disp.copy()

    # Click on fisheye region (left)
    if mx < FISH_W_DISP:
        u_full = mx * scale_fx
        v_full = my * scale_fy
        cv2.circle(fisheye_copy, (mx, my), 5, (0,255,0), -1)
        draw_text(fisheye_copy, f"F({u_full:.0f},{v_full:.0f})", (mx+6, my-6), (0,255,0))
        print(f"[click] FISHEYE display ({mx},{my}) -> full ({u_full:.1f},{v_full:.1f})")

        # map to each rectified camera
        for i, cam in enumerate(cam_settings):
            pt = fisheye_to_rect_point(u_full, v_full, cam, fisheye_params, rect_h_local, rect_w_local)
            if pt is not None:
                u_rect, v_rect = pt
                ox = (i % 2) * RECT_TILE_W_DISP
                oy = (i // 2) * RECT_TILE_H_DISP
                u_disp = int(round(u_rect * scale_rect))
                v_disp = int(round(v_rect * scale_rect))
                pos = (ox + u_disp, oy + v_disp)
                cv2.circle(rect_copy, pos, 4, (0,0,255), -1)
                draw_text(rect_copy, f"C{i+1}({u_rect},{v_rect})", (pos[0]+6, pos[1]-6), (0,0,255))
                print(f"  -> rect view {i+1}: ({u_rect},{v_rect})")

    else:
        # Click on rect-grid region (right)
        xr = mx - FISH_W_DISP
        yr = my
        if not (0 <= xr < rect_copy.shape[1] and 0 <= yr < rect_copy.shape[0]):
            print("Click outside rect grid.")
            return

        xi_disp = int(round(xr))
        yi_disp = int(round(yr))
        view_idx = (yi_disp >= RECT_TILE_H_DISP) * 2 + (xi_disp >= RECT_TILE_W_DISP)
        tile_x_disp = xi_disp % RECT_TILE_W_DISP
        tile_y_disp = yi_disp % RECT_TILE_H_DISP
        u_full = tile_x_disp / scale_rect
        v_full = tile_y_disp / scale_rect

        cv2.circle(rect_copy, (xi_disp, yi_disp), 5, (0,255,0), -1)
        draw_text(rect_copy, f"C{view_idx+1}({u_full:.0f},{v_full:.0f})", (xi_disp+6, yi_disp-6), (0,255,0))
        print(f"[click] RECT display tile {view_idx+1} display({xi_disp},{yi_disp}) -> full ({u_full:.1f},{v_full:.1f})")

        # map rect -> fisheye
        pt_f = rect_to_fisheye_point(u_full, v_full, cam_settings[view_idx], fisheye_params, rect_h_local, rect_w_local)
        if pt_f is not None:
            u_f, v_f = pt_f
            u_fd = int(round(u_f / scale_fx))
            v_fd = int(round(v_f / scale_fy))
            if 0 <= u_fd < FISH_W_DISP and 0 <= v_fd < FISH_H_DISP:
                cv2.circle(fisheye_copy, (u_fd, v_fd), 4, (255,0,0), -1)
                draw_text(fisheye_copy, f"F({u_f:.0f},{v_f:.0f})", (u_fd+6, v_fd-6), (255,0,0))

            # undistort fisheye point into rect domain, then BEV -> 3D
            pt_f_arr = np.array([[ [u_f, v_f] ]], dtype=np.float32)
            try:
                rect_point = cv2.fisheye.undistortPoints(pt_f_arr, K_fish, D_fish, P=dis_K_rect)
                mapped_pt = cv2.perspectiveTransform(rect_point, homography_BEV)
                mx_bev, my_bev = int(round(mapped_pt[0][0][0])), int(round(mapped_pt[0][0][1]))
                x_disp_bev, y_disp_bev = int(round(mx_bev * scale_BEV)), int(round(my_bev * scale_BEV))
                cv2.circle(bev_copy, (x_disp_bev, y_disp_bev), 5, (0,0,255), -1)
                draw_text(bev_copy, f"BEV({mx_bev},{my_bev})", (x_disp_bev+6, y_disp_bev-6), (0,0,255))

                p3d = map_2d_to_3d(mx_bev, my_bev, homography_BEV_to_3D)
                p3d = [float(abs(v)) for v in p3d]
                point_queue.put(p3d)
                print(f"  -> BEV ({mx_bev},{my_bev}) -> 3D {p3d}")
            except Exception as e:
                print("Error projecting fisheye->BEV:", e)

    # show annotated quick previews (non-blocking)
    preview = np.ones((max(fisheye_copy.shape[0], rect_copy.shape[0], bev_copy.shape[0]),
                       FISH_W_DISP + rect_copy.shape[1], 3), dtype=np.uint8) * 0
    preview[0:fisheye_copy.shape[0], 0:FISH_W_DISP] = fisheye_copy
    preview[0:rect_copy.shape[0], FISH_W_DISP:FISH_W_DISP + rect_copy.shape[1]] = rect_copy
    cv2.imshow("Unified Mapping (preview)", preview)
    cv2.imshow("BEV Image", bev_copy)

# Register mouse callback globally
cv2.namedWindow("Unified Mapping", cv2.WINDOW_NORMAL)
cv2.setMouseCallback("Unified Mapping", on_mouse)

# -----------------------------
# Main playback loop (thread)
# -----------------------------
def run_playback():
    caps, fisheye_cap = open_video_caps(video_paths, fisheye_path)
    try:
        while True:
            views, fisheye_frame = get_next_frames(caps, fisheye_cap)
            if any(v is None for v in views) or fisheye_frame is None:
                print("Warning: one of the frames is None; stopping playback.")
                break

            # rect frame size (assume all rect cameras same size)
            rect_h_local, rect_w_local = views[0].shape[:2]
            RECT_TILE_W_DISP = int(rect_w_local * scale_rect)
            RECT_TILE_H_DISP = int(rect_h_local * scale_rect)

            # prepare displays
            views_disp = [cv2.resize(v, (0,0), fx=scale_rect, fy=scale_rect) for v in views]
            fisheye_disp = cv2.resize(fisheye_frame, (FISH_W_DISP, FISH_H_DISP))
            bev_disp_local = cv2.resize(BEV, (0,0), fx=scale_BEV, fy=scale_BEV)

            # compose rect grid (2x2)
            grid_h = RECT_TILE_H_DISP * 2
            grid_w = RECT_TILE_W_DISP * 2
            rect_grid_disp = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)
            rect_grid_disp[0:RECT_TILE_H_DISP, 0:RECT_TILE_W_DISP] = views_disp[0]
            rect_grid_disp[0:RECT_TILE_H_DISP, RECT_TILE_W_DISP:] = views_disp[1]
            rect_grid_disp[RECT_TILE_H_DISP:, 0:RECT_TILE_W_DISP] = views_disp[2]
            rect_grid_disp[RECT_TILE_H_DISP:, RECT_TILE_W_DISP:] = views_disp[3]

            # update shared state for mouse callback
            state["rect_grid_disp"] = rect_grid_disp
            state["fisheye_disp"] = fisheye_disp
            state["bev_disp"] = bev_disp_local
            state["rect_h"] = rect_h_local
            state["rect_w"] = rect_w_local
            state["RECT_TILE_W_DISP"] = RECT_TILE_W_DISP
            state["RECT_TILE_H_DISP"] = RECT_TILE_H_DISP
            state["scale_fx"] = fisheye_params["width"] / float(FISH_W_DISP)
            state["scale_fy"] = fisheye_params["height"] / float(FISH_H_DISP)

            # compose canvas: left fisheye, right rect grid
            h_canvas = max(fisheye_disp.shape[0], rect_grid_disp.shape[0])
            w_canvas = FISH_W_DISP + rect_grid_disp.shape[1]
            canvas = np.zeros((h_canvas, w_canvas, 3), dtype=np.uint8)
            canvas[0:fisheye_disp.shape[0], 0:FISH_W_DISP] = fisheye_disp
            canvas[0:rect_grid_disp.shape[0], FISH_W_DISP:FISH_W_DISP + rect_grid_disp.shape[1]] = rect_grid_disp

            cv2.imshow("Unified Mapping", canvas)
            if cv2.waitKey(frame_delay_ms) & 0xFF == 27:
                break

    finally:
        for c in caps:
            try:
                c.release()
            except Exception:
                pass
        if fisheye_cap:
            try:
                fisheye_cap.release()
            except Exception:
                pass
        cv2.destroyAllWindows()

# start playback thread
cv_thread = threading.Thread(target=run_playback, daemon=True)
cv_thread.start()

# main loop updates Open3D from point_queue
try:
    while cv_thread.is_alive():
        while not point_queue.empty():
            p = point_queue.get()
            try:
                add_point_realtime(p)
            except Exception as e:
                print("Open3D update error:", e)
        vis.poll_events()
        vis.update_renderer()
        time.sleep(0.03)
except KeyboardInterrupt:
    pass
finally:
    try:
        vis.destroy_window()
    except Exception:
        pass
    cv2.destroyAllWindows()
    print("Clean exit.")


: 